# Statistics and Data Analysis — Homework 1

### IDs
Insert your IDs in the cell below **and** in the header of `hw1.py`.

ID #1:

ID #2:

## Read the following instructions carefully

1. This notebook contains all the instructions. Your answers are the functions in **`hw1.py`** — complete them there.
   The notebook is for you: use it to develop, test, plot and explore (add as many cells as you like).
1. **Grading is fully automatic.** The grader imports your `hw1.py` and calls every function with **hidden inputs that are
   different from the default values**, and compares the results with the correct answers.
   Each function is tested on several hidden inputs and you get partial credit for every input that is answered correctly.
   Therefore:
   * Do **not** change the names of the functions or the names/order of their parameters. You may add parameters
     at the end, with default values.
   * Every function must **return** its answer in the exact type described in its docstring (printing is not enough).
   * Your functions must work for every valid input, not only for the defaults. Hard-coded answers get no credit.
   * Unless stated otherwise, answers must be exact up to floating point accuracy (relative error below $10^{-6}$).
     **No simulations** inside graded functions (simulations in the notebook, to check yourself, are encouraged!).
   * Every function must run in a **reasonable time** (seconds, not minutes) on a standard laptop. Calls that run too long are
     stopped and get no credit — write efficient code (see the size of the inputs mentioned in each question).
   * Do not run code at the top level of `hw1.py` (only imports and function definitions), do not call `input()`,
     and use only numpy, pandas, scipy, matplotlib and the Python standard library.
1. Write **efficient, vectorized** code whenever possible.
1. Before you submit, run the **self-check** cell at the end of this notebook. It verifies that your functions run and
   return the right types (not that they are correct).
1. Submit a **zip** file that contains `hw1.py` and this notebook, named with your IDs:
   `hw1_123456789_987654321.zip` (pairs) or `hw1_123456789.zip` (single).

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

import hw1   # your solution file; thanks to autoreload, changes in hw1.py are picked up automatically

---
## Question 1 — Is the new treatment better? (20 points)

Recall the kidney cancer example from class. Under the standard of care (SoC) the 5-year survival rate is $p_0 = 0.2$.
A new experimental treatment (Tx) was given to $n = 40$ patients and $16$ of them survived 5 years.
We test $H_0$: the survival rate under Tx is $p_0$, against the alternative that it is larger.

#### 1.A (4 points)
Complete `binom_pvalue(k, n, p0)`: the right-sided p-value $P(X \ge k)$ for $X \sim Binom(n, p_0)$.

For the default values you should get the result from class, $\approx 0.002936$.

In [ ]:
hw1.binom_pvalue()

#### 1.B (4 points)
Before running a trial we fix a significance level $\alpha$ and decide to declare Tx better if $X \ge k^*$, where
$k^*$ is the **smallest** $k$ whose p-value $P(X \ge k)$ is at most $\alpha$.

Complete `critical_k(n, p0, alpha)` that returns $k^*$ (if no $k \in \{0,\dots,n\}$ qualifies, return $n+1$).

In [ ]:
hw1.critical_k()

#### Power: can the study detect a real improvement?

The rule of 1.B controls **false alarms**: if Tx is no better than SoC ($p = p_0$), the probability that we declare Tx better
($X \ge k^*$) is at most $\alpha$. But a study must also be able to **detect a real improvement**.

Suppose that the true 5-year survival rate under Tx is $p_1 > p_0$. Then $X \sim Binom(n, p_1)$, and the probability that the study
declares Tx better is called the **power** of the study:

$$\text{power}(n) = P(X \ge k^*) \quad\text{where } X \sim Binom(n, p_1) \text{ and } k^* = \texttt{critical\_k}(n, p_0, \alpha).$$

Note the two different probabilities: the threshold $k^*$ is computed under $p_0$ (when we plan the study we do not know $p_1$),
but the power is computed under $p_1$. $1 - \text{power}$ is the probability that we **miss** a real improvement.

**Example.** With $n = 40$, $p_0 = 0.2$ and $\alpha = 0.01$ we get $k^* = 15$. If the true survival rate under Tx is $p_1 = 0.35$,
the power is $P(Binom(40, 0.35) \ge 15) \approx 0.43$: such a study would *miss* a real improvement from 20% to 35% survival
more often than it would detect it!

#### 1.C (4 points)
Complete `power(n, p0, p1, alpha)`. For the default values you should get $\approx 0.4279$ (the example above).

In [ ]:
hw1.power()

#### 1.D (4 points)
Does adding a patient always increase the power? Complete `power_drops(p0, p1, alpha, n_max)` that returns, as a sorted 1-d
numpy array of ints, every $n \in \{2, \dots, n_{max}\}$ for which the power with $n$ patients is **smaller** than the power
with $n - 1$ patients.

In [ ]:
hw1.power_drops()

#### 1.E (4 points)
How many patients do we need? Complete `min_sample_size(p0, p1, alpha, target)` that returns the **smallest** $n \ge 1$
for which the power is at least `target`.

In [ ]:
hw1.min_sample_size()

---
## Question 2 — Randomistan basketball: who finishes first? (18 points)

A player shoots free throws; each shot succeeds with probability $p$, independently of all other shots.
Let $T$ be the number of shots the player takes until their $r$-th success (so $T \in \{r, r+1, \dots\}$).

> Note: `scipy.stats.nbinom` counts something slightly different. Read its documentation.

#### 2.A (6 points)
Complete `shots_pmf(t, r, p)` that returns $P(T = t)$. The argument `t` may be a single integer (return a float)
or a 1-d numpy array of integers (return an array of the same length).

In [ ]:
print(hw1.shots_pmf())
print(hw1.shots_pmf(np.arange(0, 8)))

#### 2.B (6 points)
Two players, A and B, shoot **synchronously**: in every round each of them takes one shot. All shots are independent.
Player A needs $r_A$ successes and succeeds with probability $p_A$; player B needs $r_B$ successes and succeeds with probability $p_B$.

Complete `duel_probs(pA, rA, pB, rB)` that returns the tuple
$\big(P(T_A < T_B),\ P(T_A = T_B),\ P(T_A > T_B)\big)$.

In [ ]:
hw1.duel_probs()

#### 2.C (6 points)
Complete `duel_expected_length(pA, rA, pB, rB)` that returns $E[\min(T_A, T_B)]$, the expected number of rounds until the
first of the two players completes their task.

*Hint:* for a random variable $M$ with values in $\{0, 1, 2, \dots\}$, $E[M] = \sum_{t \ge 0} P(M > t)$.

In [ ]:
hw1.duel_expected_length()

---
## Question 3 — Sums of independent random variables (17 points)

A distribution of a random variable with finitely many **integer** values is represented by a 2-d numpy array
`[[values], [probabilities]]`, e.g. a fair die is `np.array([[1, 2, 3, 4, 5, 6], [1/6]*6])`.
The values are integers, but they are not necessarily sorted, consecutive, small, or positive.

#### 3.A (8 points)
Complete `conv_pmf(P, Q)` that returns the distribution of $X + Y$, where $X \sim P$ and $Y \sim Q$ are independent,
in the same representation, with the values sorted in increasing order.

In [ ]:
die = np.array([[1, 2, 3, 4, 5, 6], [1/6]*6])
hw1.conv_pmf(die, die)

#### 3.B (9 points)
Complete `nfold_pmf(P, n)` that returns the distribution of $X_1 + \dots + X_n$ where $X_1, \dots, X_n$ are independent
and each has the distribution $P$.

Your function will be tested with $n$ up to $2000$ and with supports such as $\{0, 1000, 2500\}$ — it must be efficient.

In [ ]:
hw1.nfold_pmf(die, 3)

---
## Question 4 — Collecting coupons (17 points)

In every draw we get one of $N$ equiprobable coupons, independently of all previous draws.
Let $T_N$ be the number of draws until we have seen all $N$ coupons.

#### 4.A (8 points)
Complete `cc_cdf(N, t)` that returns $P(T_N \le t)$ **exactly** (no simulation).

Use the exact method from class (Lecture 3: $T_N$ as a sum of independent geometric waiting times).
Your function must keep a **relative** error below $10^{-6}$ for $N \le 300$ and $t \le 3000$, even when the probability is
tiny (e.g. $10^{-12}$) — closed-form formulas you may find elsewhere are not always numerically safe.

In [ ]:
hw1.cc_cdf()

#### 4.B (9 points)
Often we do not need the full collection. Complete `cc_partial_quantile(N, m, q)` that returns the smallest number of draws
$t$ such that, with probability at least $q$, at least $m$ different coupons appear among the first $t$ draws.

In [ ]:
hw1.cc_partial_quantile()

---
## Question 5 — Auditing the factory inspection logs (28 points)

The Randomistan factory has four production machines: **A, B, C and D**. Products are inspected in batches.
Every batch is inspected once, and the inspection system writes one line per batch to a log file with the columns

| column | meaning |
|---|---|
| `log_line` | line number in the log file |
| `batch_id` | identifier of the batch (every batch has a different id) |
| `timestamp` | time of the inspection |
| `machine` | the machine that produced the batch |
| `sensor_fw` | firmware version of the inspection sensor (sensors are upgraded from time to time) |
| `inspected` | number of products inspected in the batch |
| `defective` | number of defective products found in the batch |

The manufacturer of the machines claims that their defect rate is at most 3%. You are hired to audit the claim.

**Real logs are messy.** The inspection system has several kinds of recording problems, and they show up in the logs
(every log contains each kind, in different machines, at different times and to different extents).

You get three logs to explore: `factory_log_1.csv`, `factory_log_2.csv`, `factory_log_3.csv`.
Your functions will be graded on **other logs from the same system** that you do not have. They must therefore detect and
handle the problems **automatically** — do not hard-code dates, line numbers, batch ids or machines.

A batch is **usable** if its true number of defective products can be determined from the log
(possibly after correcting how it was recorded). Your analysis must use **every usable batch exactly once, with its true values,
and nothing else**.

We also give you `factory_log_clean.csv`: a log of the same system **without** recording problems. Use it to check
5.B–5.D independently of your cleaning (the expected results are given below). Your functions will also be tested on clean logs,
so make sure that your cleaning does not damage a clean log.

**Explore before you compute.**

In [ ]:
logs = {i: pd.read_csv(f'factory_log_{i}.csv') for i in [1, 2, 3]}
clean = pd.read_csv('factory_log_clean.csv')
logs[1].head()

#### 5.A (10 points)
Complete `clean_log(df)`: it receives a raw log (a DataFrame, as returned by `pd.read_csv`) and returns a DataFrame with
**one row per usable batch** and the columns `batch_id, machine, inspected, defective` holding the **true** values
(machine names exactly `'A'`, `'B'`, `'C'`, `'D'`).

Grading: for every hidden log we check the fraction of affected batches that you handled correctly;
batches without problems that you dropped or changed cost points.

In [ ]:
cleaned = {i: hw1.clean_log(logs[i]) for i in logs}
{i: len(c) for i, c in cleaned.items()}

#### 5.B (6 points)
Complete `defect_rates(df)` that returns a dict `{machine: rate}` with the estimated defect rate of each machine:
(total defective products) / (total inspected products) over the machine's usable batches.

For `factory_log_clean.csv` you should get approximately
`{'A': 0.0593801, 'B': 0.0497535, 'C': 0.0552334, 'D': 0.0241998}`.

In [ ]:
hw1.defect_rates(clean)

#### 5.C (6 points)
For a machine with $N$ inspected products and $D$ defective ones (over its usable batches), the right-sided p-value of the claim
"the defect rate is $p_0$" is $P(X \ge D)$ for $X \sim Binom(N, p_0)$.

Complete `log10_pvalue(df, machine, p0)` that returns $\log_{10}$ of this p-value.
It must be accurate (relative error below $10^{-6}$) **even when the p-value is far below $10^{-300}$**.

For `factory_log_clean.csv`: `log10_pvalue(clean, 'B', 0.03)` $\approx -829.7209$ and
`log10_pvalue(clean, 'C', 0.05)` $\approx -44.7035$.

In [ ]:
hw1.log10_pvalue(clean, 'B', 0.03), hw1.log10_pvalue(clean, 'C', 0.05)

#### 5.D (6 points)
We want to send products of a machine to a lab that needs at least $r$ defective products.
Complete `items_needed(df, machine, r, q)` that returns the smallest number of products $n$ we must take so that
$P(\text{at least } r \text{ of them are defective}) \ge q$, assuming that every product is defective independently with probability
equal to the machine's estimated defect rate (5.B).

For `factory_log_clean.csv`: `items_needed(clean, 'A', 10, 0.95)` $= 262$.

In [ ]:
hw1.items_needed(clean, 'A', 10, 0.95)

---
## Self-check (run before submitting)

This cell calls every function once and checks that it runs and returns the right **type**.
It does not check that your answers are correct.

In [ ]:
import importlib, time, traceback
importlib.reload(hw1)

def _num(x):  return isinstance(x, (int, float, np.integer, np.floating)) and not isinstance(x, bool)
def _int(x):  return isinstance(x, (int, np.integer)) and not isinstance(x, bool)
def _pmf(x):  return isinstance(x, np.ndarray) and x.ndim == 2 and x.shape[0] == 2
def _tup(n, f): return lambda x: isinstance(x, tuple) and len(x) == n and all(f(v) for v in x)
def _df(x):   return isinstance(x, pd.DataFrame) and {'batch_id', 'machine', 'inspected', 'defective'} <= set(x.columns)
def _rates(x): return isinstance(x, dict) and set(map(str, x)) == {'A', 'B', 'C', 'D'} and all(_num(v) for v in x.values())

_log = pd.read_csv('factory_log_1.csv')
_P = np.array([[0, 3, -1], [0.2, 0.5, 0.3]])
checks = [
    ('binom_pvalue',        lambda: hw1.binom_pvalue(k=5, n=20, p0=0.1),                  _num),
    ('critical_k',          lambda: hw1.critical_k(n=30, p0=0.3, alpha=0.05),            _int),
    ('power',               lambda: hw1.power(n=30, p0=0.3, p1=0.5, alpha=0.05),         _num),
    ('power_drops',         lambda: hw1.power_drops(p0=0.3, p1=0.5, alpha=0.05, n_max=40), lambda x: isinstance(x, np.ndarray) and x.ndim == 1),
    ('min_sample_size',     lambda: hw1.min_sample_size(p0=0.3, p1=0.5, alpha=0.05, target=0.7), _int),
    ('shots_pmf (int)',     lambda: hw1.shots_pmf(t=7, r=2, p=0.3),                      _num),
    ('shots_pmf (array)',   lambda: hw1.shots_pmf(t=np.arange(10), r=2, p=0.3),          lambda x: isinstance(x, np.ndarray) and x.shape == (10,)),
    ('duel_probs',          lambda: hw1.duel_probs(pA=0.2, rA=3, pB=0.3, rB=4),          _tup(3, _num)),
    ('duel_expected_length',lambda: hw1.duel_expected_length(pA=0.2, rA=3, pB=0.3, rB=4), _num),
    ('conv_pmf',            lambda: hw1.conv_pmf(P=_P, Q=_P),                            _pmf),
    ('nfold_pmf',           lambda: hw1.nfold_pmf(P=_P, n=4),                            _pmf),
    ('cc_cdf',              lambda: hw1.cc_cdf(N=10, t=30),                              _num),
    ('cc_partial_quantile', lambda: hw1.cc_partial_quantile(N=20, m=15, q=0.9),          _int),
    ('clean_log',           lambda: hw1.clean_log(df=_log.copy()),                       _df),
    ('defect_rates',        lambda: hw1.defect_rates(df=_log.copy()),                    _rates),
    ('log10_pvalue',        lambda: hw1.log10_pvalue(df=_log.copy(), machine='A', p0=0.03), _num),
    ('items_needed',        lambda: hw1.items_needed(df=_log.copy(), machine='A', r=5, q=0.9), _int),
]
n_ok = 0
for name, call, ok in checks:
    t0 = time.time()
    try:
        out = call()
        good = ok(out)
        msg = 'OK  ' if good else 'BAD TYPE: got %s' % type(out).__name__
        n_ok += good
    except Exception as e:
        msg = 'ERROR: %s: %s' % (type(e).__name__, e)
    print('%-22s %-50s %.2fs' % (name, msg, time.time() - t0))
print('\n%d / %d functions returned the expected type' % (n_ok, len(checks)))